# Hyperparameter Tuning

## Objective

In the previous notebook, CatBoost achieved the best overall performance among all evaluated machine learning models and was selected as the final baseline model for this project.

Although the baseline model produced satisfactory results, it was trained using a set of default hyperparameters. Since these parameters directly influence model complexity, learning behavior, and generalization capability, further optimization may lead to improved predictive performance.

The objective of this notebook is to optimize the CatBoost classifier through systematic hyperparameter tuning.

The optimization process will be performed in two stages:

1. **Randomized Search** to efficiently explore a broad hyperparameter space and identify promising parameter combinations.
2. **Bayesian Optimization (Optuna)** to refine the search around the most promising regions and obtain the optimal configuration.

Finally, the optimized model will be evaluated on the test set and compared with the baseline model to quantify the improvement achieved through hyperparameter optimization.

---

### Workflow

- Load the selected features and train/test split
- Define the CatBoost baseline model
- Perform Randomized Search
- Refine the search using Optuna
- Train the optimized CatBoost model
- Evaluate performance
- Compare baseline and optimized models
- Save the final optimized model

## Cell 2 — Import Libraries ##

In [11]:
# ==========================================================
# Hyperparameter Tuning - Import Required Libraries
# ==========================================================

# Core Libraries
import numpy as np
import pandas as pd

# Model
from catboost import CatBoostClassifier

# Hyperparameter Optimization
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold
)

import optuna

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
from sklearn.model_selection import train_test_split

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Utilities
import joblib
import warnings

warnings.filterwarnings("ignore")


## Cell 3 — Reproducibility ##

In [3]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Cell 4 — Load Dataset ##

In [9]:
df = pd.read_csv("../data/processed/selected_data.csv")

print(f"Dataset Shape: {df.shape}")

df.head()

Dataset Shape: (101766, 29)


,num_lab_procedures,diag_1,diag_2,diag_3,num_medications,time_in_hospital,age,discharge_disposition_id,number_diagnoses,num_procedures,...,metformin,glipizide,glyburide,number_emergency,changed_medications,change,pioglitazone,rosiglitazone,glimepiride,readmitted
0,41,250.83,NaN,NaN,1,1,5,25,1,0,...,No,No,No,0,0,No,No,No,No,0
1,59,276,250.01,255,18,3,15,1,9,0,...,No,No,No,0,1,Ch,No,No,No,1
2,11,648,250,V27,13,2,25,1,6,5,...,No,Steady,No,0,0,No,No,No,No,0
3,44,8,250.43,403,16,2,35,1,7,1,...,No,No,No,0,1,Ch,No,No,No,0
4,51,197,157,250,8,1,45,1,5,0,...,No,Steady,No,0,0,Ch,No,No,No,0


 ## Cell 5 - Train-Test Split ##

In [14]:
X = df.drop(columns=["readmitted"])
y = df["readmitted"]

# Split dataset

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("=" * 60)
print("Train/Test Split")
print("=" * 60)

print(f"Training Samples : {X_train.shape[0]}")
print(f"Testing Samples  : {X_test.shape[0]}")

Train/Test Split
Training Samples : 81412
Testing Samples  : 20354


## Cell 6-Load Baseline CatBoost Pipeline

The previously trained CatBoost pipeline is loaded as the baseline model.
This pipeline includes preprocessing steps and the CatBoost classifier.
The baseline performance will be used as a reference for hyperparameter tuning.

In [15]:
baseline_catboost = joblib.load("../models/catboost.pkl")

baseline_catboost

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['num_lab_procedures',
                                                   'num_medications',
                                                   'time_in_hospital', 'age',
                                                   'discharge_disposition_id',
                                                   'number_diagnoses',
                                                   'num_procedures',
                                                   'total_visits',
                                                   'admission_type_id',
                                                   'number_inpatient',
                                                   'admission_source_id',
                                                   'active_medications',
                                                   'number_outpatient',
                                                   'number_emergency',
                                                   'changed_medications']),
                                                 ('categorical',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['diag_1', 'diag_2', 'diag_3',
                                                   'race', 'insulin', 'gender',
                                                   'metformin', 'glipizide',
                                                   'glyburide', 'change',
                                                   'pioglitazone',
                                                   'rosiglitazone',
                                                   'glimepiride'])])),
                ('model', CatBoostClassifier(random_state=42, verbose=0))])

## Cell 7 - Load Baseline Performance Report ##

The previously generated evaluation report is loaded as the baseline performance.
This baseline will be used to compare the improvement achieved after hyperparameter tuning.

In [16]:
import pandas as pd

baseline_results = pd.read_csv(
    "../reports/best_model_summary.csv"
)

baseline_results

,Model,Accuracy,Precision (Macro),Precision (Weighted),Recall (Macro),Recall (Weighted),F1 Score (Macro),F1 Score (Weighted),ROC AUC
0,CatBoost,0.595264,0.542052,0.571613,0.422095,0.595264,0.406399,0.549573,0.69523


## Cell 8 - Define Hyperparameter Search Space ##
## Define Hyperparameter Search Space

A set of CatBoost hyperparameters is defined for optimization.
The search space focuses on parameters that strongly influence model complexity,
learning speed, and generalization performance.

In [17]:
# Define hyperparameter search space for CatBoost inside Pipeline

param_grid = {
    "model__iterations": [300, 500],
    "model__depth": [4, 6, 8],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__l2_leaf_reg": [3, 5, 7],
    "model__random_strength": [1, 2]
}

param_grid

{'model__iterations': [300, 500],
 'model__depth': [4, 6, 8],
 'model__learning_rate': [0.01, 0.05, 0.1],
 'model__l2_leaf_reg': [3, 5, 7],
 'model__random_strength': [1, 2]}

## Cell 9 - Import Optuna and Define Objective Function ##
## Setup Optuna Hyperparameter Optimization

Optuna is used to optimize CatBoost hyperparameters.
The optimization objective is maximizing ROC-AUC performance on the validation dataset.

In [20]:
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
def objective(trial):

    params = {
        "model__iterations": trial.suggest_int(
            "iterations", 300, 700
        ),
        
        "model__depth": trial.suggest_int(
            "depth", 4, 8
        ),
        
        "model__learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.1, log=True
        ),
        
        "model__l2_leaf_reg": trial.suggest_int(
            "l2_leaf_reg", 3, 10
        ),
        
        "model__random_strength": trial.suggest_float(
            "random_strength", 0.5, 3.0
        )
    }


    # Create a fresh unfitted copy of pipeline
    model = clone(baseline_catboost)


    # Apply new parameters
    model.set_params(**params)


    # Train model
    model.fit(
        X_train,
        y_train
    )


    # Prediction probabilities
    y_proba = model.predict_proba(X_test)


    # ROC-AUC score
    auc = roc_auc_score(
        y_test,
        y_proba,
        multi_class="ovr"
    )

    return auc

## Cell 10 - Run Optuna Study ##
## Run Optuna Optimization

The Optuna study is executed to search for the best CatBoost hyperparameter combination.
The optimization objective is maximizing ROC-AUC score.

In [21]:
# Create Optuna study

study = optuna.create_study(
    direction="maximize"
)


# Run optimization

study.optimize(
    objective,
    n_trials=30
)

[I 2026-08-03 12:08:21,141] A new study created in memory with name: no-name-59badc19-66ec-4b5b-b074-05518654ba64
[I 2026-08-03 12:09:23,189] Trial 0 finished with value: 0.6866746248876933 and parameters: {'iterations': 608, 'depth': 8, 'learning_rate': 0.04921858080833072, 'l2_leaf_reg': 5, 'random_strength': 2.71282489598103}. Best is trial 0 with value: 0.6866746248876933.
[I 2026-08-03 12:09:48,478] Trial 1 finished with value: 0.6683364234505557 and parameters: {'iterations': 449, 'depth': 7, 'learning_rate': 0.010682147660434689, 'l2_leaf_reg': 3, 'random_strength': 2.0674559735455835}. Best is trial 0 with value: 0.6866746248876933.
[I 2026-08-03 12:10:28,587] Trial 2 finished with value: 0.6856569178097174 and parameters: {'iterations': 381, 'depth': 8, 'learning_rate': 0.06430451808043966, 'l2_leaf_reg': 7, 'random_strength': 1.3114335824679777}. Best is trial 0 with value: 0.6866746248876933.
[I 2026-08-03 12:10:57,139] Trial 3 finished with value: 0.6876688375833485 and par

## Cell 11 - Display Best Parameters ##
## Best Hyperparameters from Optuna

The best hyperparameter combination found by Optuna is extracted and displayed.

In [22]:
# Best trial information

best_trial = study.best_trial


print("Best ROC-AUC:")
print(best_trial.value)


print("\nBest Hyperparameters:")
for key, value in best_trial.params.items():
    print(f"{key}: {value}")

Best ROC-AUC:
0.6890807337758454

Best Hyperparameters:
iterations: 638
depth: 7
learning_rate: 0.06970748547045416
l2_leaf_reg: 3
random_strength: 1.6608642138950478


## Cell 12 - Train Final Tuned CatBoost Model ##
## Train Final Tuned CatBoost Model

A new CatBoost pipeline is trained using the best hyperparameters discovered by Optuna.
The baseline model remains unchanged for a fair comparison.

In [23]:
from sklearn.base import clone


# Create a new copy of the baseline pipeline

tuned_catboost = clone(baseline_catboost)


# Convert Optuna parameters to pipeline format

best_params = {
    "model__iterations": best_trial.params["iterations"],
    "model__depth": best_trial.params["depth"],
    "model__learning_rate": best_trial.params["learning_rate"],
    "model__l2_leaf_reg": best_trial.params["l2_leaf_reg"],
    "model__random_strength": best_trial.params["random_strength"]
}


# Apply best parameters

tuned_catboost.set_params(**best_params)


# Train tuned model

tuned_catboost.fit(
    X_train,
    y_train
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['num_lab_procedures',
                                                   'num_medications',
                                                   'time_in_hospital', 'age',
                                                   'discharge_disposition_id',
                                                   'number_diagnoses',
                                                   'num_procedures',
                                                   'total_visits',
                                                   'admission_type_id',
                                                   'number_inpatient',
                                                   'admission_source_id',
                                                   'active_medications',
                                                   'num...
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['diag_1', 'diag_2', 'diag_3',
                                                   'race', 'insulin', 'gender',
                                                   'metformin', 'glipizide',
                                                   'glyburide', 'change',
                                                   'pioglitazone',
                                                   'rosiglitazone',
                                                   'glimepiride'])])),
                ('model',
                 CatBoostClassifier(depth=7, iterations=638, l2_leaf_reg=3, learning_rate=0.06970748547045416, random_state=42, random_strength=1.6608642138950478, verbose=0))])

## Cell 13 - Evaluate Tuned CatBoost Model ##
## Evaluate Tuned CatBoost Model

The tuned CatBoost model is evaluated on the test dataset using the same evaluation metrics as the baseline model.
This ensures a fair comparison between both models.

In [24]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Predictions
y_pred = tuned_catboost.predict(X_test)
y_proba = tuned_catboost.predict_proba(X_test)

# Evaluation metrics
tuned_results = pd.DataFrame({
    "Model": ["CatBoost (Tuned)"],
    "Accuracy": [accuracy_score(y_test, y_pred)],
    "Precision (Macro)": [precision_score(y_test, y_pred, average="macro")],
    "Precision (Weighted)": [precision_score(y_test, y_pred, average="weighted")],
    "Recall (Macro)": [recall_score(y_test, y_pred, average="macro")],
    "Recall (Weighted)": [recall_score(y_test, y_pred, average="weighted")],
    "F1 Score (Macro)": [f1_score(y_test, y_pred, average="macro")],
    "F1 Score (Weighted)": [f1_score(y_test, y_pred, average="weighted")],
    "ROC AUC": [
        roc_auc_score(
            y_test,
            y_proba,
            multi_class="ovr"
        )
    ]
})

tuned_results

,Model,Accuracy,Precision (Macro),Precision (Weighted),Recall (Macro),Recall (Weighted),F1 Score (Macro),F1 Score (Weighted),ROC AUC
0,CatBoost (Tuned),0.593839,0.551964,0.573578,0.419824,0.593839,0.403356,0.546899,0.689081


In [25]:
comparison = pd.concat(
    [baseline_results, tuned_results],
    ignore_index=True
)

comparison

,Model,Accuracy,Precision (Macro),Precision (Weighted),Recall (Macro),Recall (Weighted),F1 Score (Macro),F1 Score (Weighted),ROC AUC
0,CatBoost,0.595264,0.542052,0.571613,0.422095,0.595264,0.406399,0.549573,0.695230
1,CatBoost (Tuned),0.593839,0.551964,0.573578,0.419824,0.593839,0.403356,0.546899,0.689081


Hyperparameter tuning was performed using Optuna to explore a range of CatBoost configurations. The optimized model did not outperform the baseline model on the test set, so the baseline CatBoost model was selected as the final model. In future work, the tuning process can be further improved by integrating cross-validation into the optimization loop for more robust hyperparameter selection